In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import accuracy_score, confusion_matrix
from utils import NoisyFashionMNIST

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

transform = transforms.Compose([transforms.ToTensor()])
train_cls = datasets.FashionMNIST("./data", train=True, download=True, transform=transform)
test_cls = datasets.FashionMNIST("./data", train=False, download=True, transform=transform)
train_cls_loader = DataLoader(train_cls, batch_size=64, shuffle=True)
test_cls_loader = DataLoader(test_cls, batch_size=1000, shuffle=False)

train_denoise = NoisyFashionMNIST("./data", True)
test_denoise = NoisyFashionMNIST("./data", False)
train_denoise_loader = DataLoader(train_denoise, batch_size=64, shuffle=True)
test_denoise_loader = DataLoader(test_denoise, batch_size=64, shuffle=False)

# baseline, batchnorm, dropout, residual
class CNN_Baseline(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 64 * 7 * 7)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

class CNN_Dropout(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 64 * 7 * 7)
        x = self.dropout(F.relu(self.fc1(x)))
        x = self.fc2(x)
        return x

class CNN_BatchNorm(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = x.view(-1, 64 * 7 * 7)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# denoising models: include Dropout and support different loss functions
class DenoisingAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),
            nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(64, 32, 3, stride=2, output_padding=1, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 1, 3, stride=2, output_padding=1, padding=1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        x = self.encoder(x)
        x = self.decoder(x)
        return x

# support different loss functions for denoising
from torchmetrics.functional import structural_similarity_index_measure as ssim

def train_denoising(model, loss_fn='mse', epochs=5):
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for noisy, clean in train_denoise_loader:
            noisy, clean = noisy.to(device), clean.to(device)
            out = model(noisy)

            if loss_fn == 'mse':
                loss = F.mse_loss(out, clean)
            elif loss_fn == 'mae':
                loss = F.l1_loss(out, clean)
            elif loss_fn == 'ssim':
                loss = 1 - ssim(out, clean)  # SSIM more is better, so we minimize 1 - SSIM

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        print(f"[{loss_fn.upper()}] Epoch {epoch+1}: Loss={total_loss/len(train_denoise_loader):.4f}")

# General classifier training and testing functions
def train_and_eval_classification(model, epochs=5):
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()
        for x, y in train_cls_loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss = criterion(out, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for x, y in test_cls_loader:
            x = x.to(device)
            pred = model(x).argmax(dim=1).cpu()
            all_preds.extend(pred)
            all_labels.extend(y)
    acc = accuracy_score(all_labels, all_preds)
    print(f"Test Accuracy: {acc:.4f}")
    return acc

print("\n--- Classification Experiments ---")
train_and_eval_classification(CNN_Baseline())
train_and_eval_classification(CNN_BatchNorm())
train_and_eval_classification(CNN_Dropout())

print("\n--- Denoising Experiments ---")
train_denoising(DenoisingAutoencoder(), loss_fn='mse')
train_denoising(DenoisingAutoencoder(), loss_fn='mae')
train_denoising(DenoisingAutoencoder(), loss_fn='ssim')



--- Classification Experiments ---
Test Accuracy: 0.9085
Test Accuracy: 0.9152
Test Accuracy: 0.9072

--- Denoising Experiments ---
[MSE] Epoch 1: Loss=0.0094
[MSE] Epoch 2: Loss=0.0031
[MSE] Epoch 3: Loss=0.0029
[MSE] Epoch 4: Loss=0.0028
[MSE] Epoch 5: Loss=0.0027
[MAE] Epoch 1: Loss=0.0547
[MAE] Epoch 2: Loss=0.0329
[MAE] Epoch 3: Loss=0.0309
[MAE] Epoch 4: Loss=0.0300
[MAE] Epoch 5: Loss=0.0294


/Users/yahuanshi/.pyenv/versions/3.12.4/lib/python3.12/site-packages/torchmetrics/utilities/prints.py:70: FutureWarning: Importing `spectral_angle_mapper` from `torchmetrics.functional` was deprecated and will be removed in 2.0. Import `spectral_angle_mapper` from `torchmetrics.image` instead.
  _future_warning(


[SSIM] Epoch 1: Loss=0.1599
[SSIM] Epoch 2: Loss=0.0846
[SSIM] Epoch 3: Loss=0.0803
[SSIM] Epoch 4: Loss=0.0780
[SSIM] Epoch 5: Loss=0.0763
